In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Simulator, scan
from mxlpy.parallel import parallelise
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

from mxlmodels import get_lazar1997

# this model needs assimulo for proper integration

## Figure 2

In [ ]:
res_fig2 = {}

res_fig2["results"] = (
    Simulator(get_lazar1997())
    .simulate_time_course(np.linspace(0.5e-4, 1, int(1e5)))
    .get_result()
    .unwrap_or_err()
    .get_combined()
    .iloc[1:]
)

idx_O = np.argmin(res_fig2["results"]["F"])
idx_P = np.argmax(res_fig2["results"]["F"])

# Find J and I
log_time = np.log10(res_fig2["results"].index)
smoothed_f = gaussian_filter1d(res_fig2["results"]["F"], sigma=3)
derivative = np.gradient(smoothed_f, log_time)
inverted_derivative = -derivative
valleys, _ = find_peaks(inverted_derivative)
valid_valleys = [v for v in valleys if idx_O < v < idx_P]

idx_J = valid_valleys[0]
idx_I = valid_valleys[1]

res_fig2["OJIP"] = {
    "O": idx_O,
    "J": idx_J,
    "I": idx_I,
    "P": idx_P
}

In [ ]:
fig, ax = plt.subplots()

to_plot = {
    "F": {},
    "y2": {"label": r"$\mathrm{Q_A^-Q_B}$", "ls": "-"},
    "y4": {"label": r"$\mathrm{Q_A^-Q_B^-}$", "ls": "-"},
    "y6": {"label": r"$\mathrm{Q_A^-Q_B^{2-}}$", "ls": (0, (3, 1, 1, 1, 1, 1))},
    "y8": {"label": r"$\mathrm{Q_A^-Q_BH_2}$", "ls": "--"},
    "y14": {"label": r"$\mathrm{Q_A^-N}$", "ls": (0, (1, 10))},
    "y15": {"label": r"$\mathrm{Q_A^-I}$", "ls": ":"},
    "y12": {"label": r"$\mathrm{Q_A^-R}$", "ls": (0, (3, 5, 1, 5))},
}

for var, dic in to_plot.items():
    ax.plot(res_fig2["results"][var], color="black", **dic)

for letter, idx in res_fig2["OJIP"].items():
    ax.text(
        x=res_fig2["results"].index[idx],
        y=res_fig2["results"]["F"].iloc[idx] + 0.02,
        s=letter,
        ha="center",
        va="bottom"
    )

ax.legend(frameon=False)
ax.set_xscale("log")
ax.set_xlim(1e-5, 0.5e1)
ax.set_xticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0])
ax.set_ylim(-0.01, 1)

plt.tight_layout()
plt.show()

## Figure 5

In [ ]:
factors = [0, 0.9, 0.77, 0.5, 1]

res_fig3 = {}
res_fig3["results"] = scan.time_course(
    model=get_lazar1997(),
    time_points=np.linspace(0.5e-4, 1, int(1e5)),
    to_scan=pd.DataFrame({
        "k6": [3500 * (1 - a) for a in factors],
        "k7": [175 * (1 - a) for a in factors],
        "k8": [1750 * (1 - a) for a in factors],
        "k9": [35 * (1 - a) for a in factors],
        "k16": [150 * (1 - a) for a in factors],
        "k17": [100 * (1 - a) for a in factors],
        "k22": [150 * (1 - a) for a in factors],
        "k23": [100 * (1 - a) for a in factors]
    }, index=factors)
).combined
mask = res_fig3["results"].groupby(level=0).cumcount() > 0
res_fig3["results"] = res_fig3["results"].loc[mask]

idx_O = np.argmin(res_fig3["results"].loc[0]["F"])
idx_P = np.argmax(res_fig3["results"].loc[0]["F"])

# Find J and I
log_time = np.log10(res_fig3["results"].loc[0]["F"].index)
smoothed_f = gaussian_filter1d(res_fig3["results"].loc[0]["F"], sigma=3)
derivative = np.gradient(smoothed_f, log_time)
inverted_derivative = -derivative
valleys, _ = find_peaks(inverted_derivative)
valid_valleys = [v for v in valleys if idx_O < v < idx_P]

idx_J = valid_valleys[0]
idx_I = valid_valleys[1]

res_fig3["OJIP"] = {
    "O": idx_O,
    "J": idx_J,
    "I": idx_I,
    "P": idx_P
}

res_fig3["rF"] = {
    "rFJ": {},
    "rFI": {}
}
for factor in factors:
    res = res_fig3["results"].loc[factor]
    FO = res["F"].iloc[idx_O]
    FJ = res["F"].iloc[idx_J]
    FI = res["F"].iloc[idx_I]
    FP = res["F"].iloc[idx_P]
    res_fig3["rF"]["rFI"][factor] = (FI - FO) / (FP - FO)
    res_fig3["rF"]["rFJ"][factor] = (FJ - FO) / (FP - FO)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ls_dict = {
    0: "-",
    0.5: "--",
    0.77: ":",
    0.9: "-.",
    1: (0, (3, 1, 1, 1, 1, 1))
}

for a in factors:
    res = res_fig3["results"].loc[a]
    ax.plot(res["F"], color="black", label=f"{a * 100:.0f}%", linestyle=ls_dict.get(a, "-"))

for letter, idx in res_fig3["OJIP"].items():
    res = res_fig3["results"].loc[0]
    ax.text(
        x=res.index[idx],
        y=res["F"].iloc[idx] + 0.02,
        s=letter,
        ha="center",
        va="bottom"
    )

ax.legend(frameon=False, title="decrease of the rate constants [%]", loc="upper left")
ax.set_xscale("log")
ax.set_xlim(1e-5, 0.5e1)
ax.set_xticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0])
ax.set_xlabel("time [s]")
ax.set_ylim(-0.01, 1)
ax.set_ylabel("fluorescence intensity [rel. units]")

ax_inset = ax.inset_axes([0.6, 0.1, 0.35, 0.35])

for rF in res_fig3["rF"]:
    marker = "s" if rF == "rFJ" else "o"
    res = dict(sorted(res_fig3["rF"][rF].items()))
    ax_inset.plot(np.array(list(res.keys())) * 100, list(res.values()), marker=marker, color="black", markersize=3.5, label=rF)

ax_inset.set_xlim(-10, 110)
ax_inset.set_xticks(np.linspace(0, 100, 6))
ax_inset.set_xlabel("decrease of the rate constants [%]")
ax_inset.set_ylim(0.3, 1.1)
ax_inset.set_yticks(np.linspace(0.4, 1.0, 4))
ax_inset.set_ylabel(r"$\mathrm{rF_J}$, $\mathrm{rF_I}$")
ax_inset.legend(frameon=False)

plt.tight_layout()
plt.show()

## Figure 6

In [ ]:
amounts = [
    ("0", 0),
    ("0.25", 0.25),
    ("0.5", 0.5),
    ("0.75", 0.75),
    ("1.0", 1.0)
]
amounts_str = [am for am, _ in amounts]
amounts_float = [am for _, am in amounts]

old_y13 = get_lazar1997().get_raw_variables()["y13"].initial_value
old_y1 = get_lazar1997().get_raw_variables()["y1"].initial_value
old_y15 = get_lazar1997().get_raw_variables()["y15"].initial_value

def simulate_for_amount(am):
    new_y0 = {
        "y13": (1 - old_y15) * am,
        "y1": (1 - old_y15) * (1 - am),
        "y15": old_y15
    }
    return (
        Simulator(get_lazar1997())
        .update_variables(new_y0)
        .simulate_time_course(np.linspace(0.5e-4, 1, int(1e5)))
        .get_result()
        .unwrap_or_err()
        .get_combined()
        .iloc[1:]
    )

res = parallelise(simulate_for_amount, amounts)
res_fig6 = {"results": {}}
for am, r in res:
    res_fig6["results"][am] = r

idx_O = np.argmin(res_fig6["results"][amounts_str[1]]["F"])
idx_P = np.argmax(res_fig6["results"][amounts_str[1]]["F"])

# Find J and I
log_time = np.log10(res_fig6["results"][amounts_str[1]]["F"].index)
smoothed_f = gaussian_filter1d(res_fig6["results"][amounts_str[1]]["F"], sigma=3)
derivative = np.gradient(smoothed_f, log_time)
inverted_derivative = -derivative
valleys, _ = find_peaks(inverted_derivative)
valid_valleys = [v for v in valleys if idx_O < v < idx_P]

idx_J = valid_valleys[0]
idx_I = valid_valleys[1]

res_fig6["OJIP"] = {
    "O": idx_O,
    "J": idx_J,
    "I": idx_I,
    "P": idx_P
}

res_fig6["rF"] = {
    "rFJ": {},
    "rFI": {}
}
for am in amounts_str:
    res = res_fig6["results"][am]
    FO = res["F"].iloc[idx_O]
    FJ = res["F"].iloc[idx_J]
    FI = res["F"].iloc[idx_I]
    FP = res["F"].iloc[idx_P]
    res_fig6["rF"]["rFI"][am] = (FI - FO) / (FP - FO)
    res_fig6["rF"]["rFJ"][am] = (FJ - FO) / (FP - FO)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ls_dict = {
    amounts_str[0]: "-",
    amounts_str[1]: "--",
    amounts_str[2]: ":",
    amounts_str[3]: "-.",
    amounts_str[4]: (0, (3, 1, 1, 1, 1, 1))
}

for am_str, am_float in amounts[::-1]:
    res = res_fig6["results"][am_str]
    ax.plot(res["F"], label=f"{am_float * 100:.0f}%", linestyle=ls_dict.get(am_str, "-"), color="black")

for letter, idx in res_fig3["OJIP"].items():
    res = res_fig6["results"][amounts_str[1]]
    ax.text(
        x=res.index[idx],
        y=res["F"].iloc[idx] + 0.02,
        s=letter,
        ha="center",
        va="bottom"
    )

ax.legend(frameon=False, title="decrease of the rate constants [%]", loc="upper left")
ax.set_xscale("log")
ax.set_xlim(1e-5, 0.5e1)
ax.set_xticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0])
ax.set_xlabel("time [s]")
ax.set_ylim(-0.01, 0.95)
ax.set_ylabel("fluorescence intensity [rel. units]")

ax_inset = ax.inset_axes([0.6, 0.1, 0.35, 0.35])

for rF in res_fig3["rF"]:
    marker = "s" if rF == "rFJ" else "o"
    res = dict(sorted(res_fig6["rF"][rF].items()))
    ax_inset.plot(np.array(list(float(am) for am in res.keys())) * 100, list(res.values()), marker=marker, color="black", markersize=3.5, label=rF)

ax_inset.set_xlim(-10, 110)
ax_inset.set_xticks(np.linspace(0, 100, 6))
ax_inset.set_xlabel("decrease of the rate constants [%]")
ax_inset.set_ylim(0.1, 1.1)
ax_inset.set_yticks(np.linspace(0.2, 1.0, 5))
ax_inset.set_ylabel(r"$\mathrm{rF_J}$, $\mathrm{rF_I}$")
ax_inset.legend(frameon=False)

plt.tight_layout()
plt.show()

## Figure 7

In [ ]:
factors_fig7 = [0, 0.1, 0.2, 0.4]

parameters_to_change = ["k1", "k2", "k3", "k4", "k5"]
parameters_to_change = {param: [get_lazar1997().get_parameter_values()[param] * (1 - a) for a in factors_fig7] for param in parameters_to_change}

res_fig7 = {}
res_fig7["results"] = scan.time_course(
    model=get_lazar1997(),
    time_points=np.linspace(0.5e-4, 1, int(1e5)),
    to_scan=pd.DataFrame(parameters_to_change, index=factors_fig7)
).combined
mask = res_fig7["results"].groupby(level=0).cumcount() > 0
res_fig7["results"] = res_fig7["results"].loc[mask]

idx_O = np.argmin(res_fig7["results"].loc[0]["F"])
idx_P = np.argmax(res_fig7["results"].loc[0]["F"])

# Find J and I
log_time = np.log10(res_fig7["results"].loc[0]["F"].index)
smoothed_f = gaussian_filter1d(res_fig7["results"].loc[0]["F"], sigma=3)
derivative = np.gradient(smoothed_f, log_time)
inverted_derivative = -derivative
valleys, _ = find_peaks(inverted_derivative)
valid_valleys = [v for v in valleys if idx_O < v < idx_P]

idx_J = valid_valleys[0]
idx_I = valid_valleys[1]

res_fig7["OJIP"] = {
    "O": idx_O,
    "J": idx_J,
    "I": idx_I,
    "P": idx_P
}

res_fig7["rF"] = {
    "rFJ": {},
    "rFI": {}
}
for factor in factors_fig7:
    res = res_fig7["results"].loc[factor]
    FO = res["F"].iloc[idx_O]
    FJ = res["F"].iloc[idx_J]
    FI = res["F"].iloc[idx_I]
    FP = res["F"].iloc[idx_P]
    res_fig7["rF"]["rFI"][factor] = (FI - FO) / (FP - FO)
    res_fig7["rF"]["rFJ"][factor] = (FJ - FO) / (FP - FO)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

ls_dict = {
    factors_fig7[0]: "-",
    factors_fig7[1]: "--",
    factors_fig7[2]: ":",
    factors_fig7[3]: "-.",
}

for a in factors_fig7:
    res = res_fig7["results"].loc[a]
    ax.plot(res["F"], color="black", label=f"{a * 100:.0f}%", linestyle=ls_dict.get(a, "-"))

for letter, idx in res_fig7["OJIP"].items():
    res = res_fig7["results"].loc[0]
    ax.text(
        x=res.index[idx],
        y=res["F"].iloc[idx] + 0.02,
        s=letter,
        ha="center",
        va="bottom"
    )

ax.legend(frameon=False, title="decrease of the rate constants [%]", loc="upper left")
ax.set_xscale("log")
ax.set_xlim(1e-5, 0.5e1)
ax.set_xticks([1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1e0])
ax.set_xlabel("time [s]")
ax.set_ylim(0, 0.85)
ax.set_yticks(np.linspace(0, 0.8, 5))
ax.set_ylabel("fluorescence intensity [rel. units]")

ax_inset = ax.inset_axes([0.6, 0.1, 0.35, 0.35])

for rF in res_fig7["rF"]:
    marker = "s" if rF == "rFJ" else "o"
    res = dict(sorted(res_fig7["rF"][rF].items()))
    ax_inset.plot(np.array(list(res.keys())) * 100, list(res.values()), marker=marker, color="black", markersize=3.5, label=rF)

ax_inset.set_xlim(-5, 50)
ax_inset.set_xticks(np.linspace(0, 40, 5))
ax_inset.set_xlabel("decrease of the rate constants [%]")
ax_inset.set_ylim(0.2, 0.7)
ax_inset.set_yticks(np.linspace(0.2, 0.6, 3))
ax_inset.set_ylabel(r"$\mathrm{rF_J}$, $\mathrm{rF_I}$")
ax_inset.legend(frameon=False)

plt.tight_layout()
plt.show()

## Figure 8

In [ ]:
old_y15 = get_lazar1997().get_raw_variables()["y15"].initial_value

settings = [
    ("0", {
        "y13": (1 - old_y15) * 0.25,
        "y1": (1 - old_y15) * (1 - 0.25),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 1,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 1,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 1,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 1,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 1
    }),
    ("1", {
        "y13": (1 - old_y15) * 0.25,
        "y1": (1 - old_y15) * (1 - 0.25),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.9,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.9,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.9,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.9,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.9
    }),
    ("2", {
        "y13": (1 - old_y15) * 0.25,
        "y1": (1 - old_y15) * (1 - 0.25),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
    ("3", {
        "y13": (1 - old_y15) * 0.33,
        "y1": (1 - old_y15) * (1 - 0.33),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
    ("4", {
        "y13": (1 - old_y15) * 0.5,
        "y1": (1 - old_y15) * (1 - 0.5),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
    ("5", {
        "y13": (1 - old_y15) * 0.67,
        "y1": (1 - old_y15) * (1 - 0.67),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
    ("6", {
        "y13": (1 - old_y15) * 0.75,
        "y1": (1 - old_y15) * (1 - 0.75),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
    ("7", {
        "y13": (1 - old_y15) * 1,
        "y1": (1 - old_y15) * (1 - 1),
        "k1": get_lazar1997().get_parameter_values()["k1"] * 0.8,
        "k2": get_lazar1997().get_parameter_values()["k2"] * 0.8,
        "k3": get_lazar1997().get_parameter_values()["k3"] * 0.8,
        "k4": get_lazar1997().get_parameter_values()["k4"] * 0.8,
        "k5": get_lazar1997().get_parameter_values()["k5"] * 0.8
    }),
]
degrees = [degree for degree, _ in settings]

def simulate_for_amount(setting):
    return (
        Simulator(get_lazar1997())
        .update_variables({
            "y13": setting["y13"],
            "y1": setting["y1"]
        })
        .update_parameters({
            "k1": setting["k1"],
            "k2": setting["k2"],
            "k3": setting["k3"],
            "k4": setting["k4"],
            "k5": setting["k5"]
        })
        .simulate_time_course(np.linspace(0.5e-4, 1, int(1e5)))
        .get_result()
        .unwrap_or_err()
        .get_combined()
        .iloc[1:]
    )

res = parallelise(simulate_for_amount, settings)
res_fig8 = {"results": {}}
for degree, r in res:
    res_fig8["results"][degree] = r

idx_O = np.argmin(res_fig8["results"][degrees[0]]["F"])
idx_P = np.argmax(res_fig8["results"][degrees[0]]["F"])

# Find J and I
log_time = np.log10(res_fig6["results"][degrees[0]]["F"].index)
smoothed_f = gaussian_filter1d(res_fig6["results"][degrees[0]]["F"], sigma=3)
derivative = np.gradient(smoothed_f, log_time)
inverted_derivative = -derivative
valleys, _ = find_peaks(inverted_derivative)
valid_valleys = [v for v in valleys if idx_O < v < idx_P]

idx_J = valid_valleys[0]
idx_I = valid_valleys[1]

res_fig8["OJIP"] = {
    "O": idx_O,
    "J": idx_J,
    "I": idx_I,
    "P": idx_P
}

res_fig8["rF"] = {
    "rFJ": {},
    "rFI": {}
}
for degree in degrees:
    res = res_fig8["results"][degree]
    FO = res["F"].iloc[idx_O]
    FJ = res["F"].iloc[idx_J]
    FI = res["F"].iloc[idx_I]
    FP = res["F"].iloc[idx_P]
    res_fig8["rF"]["rFI"][degree] = (FI - FO) / (FP - FO)
    res_fig8["rF"]["rFJ"][degree] = (FJ - FO) / (FP - FO)

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))

for rF in res_fig8["rF"]:
    marker = "s" if rF == "rFJ" else "o"
    res = dict(sorted(res_fig8["rF"][rF].items()))
    ax.plot([int(k) for k in res.keys()], list(res.values()), marker=marker, color="black", markersize=3.5, label=rF)

ax.set_xlim(-0.5, 7.5)
ax.set_xticks(np.linspace(0, 7, 8))
ax.set_xlabel("degree of damage by herbicide action")
ax.set_ylim(0.3, 1.1)
ax.set_yticks(np.linspace(0.4, 1.0, 4))
ax.set_ylabel(r"$\mathrm{rF_J}$, $\mathrm{rF_I}$")
ax.legend(frameon=False)

plt.tight_layout()
plt.show()